# Skin-Lesion Classification on HAM10000: CNN Baseline, Transfer Learning and a Critical Comparison with DiffMIC

**CM4126 Computer Vision, Coursework 2026/27** | Dataset: [HAM10000](https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000) | Reviewed paper: *[DiffMIC: Dual-Guidance Diffusion Network for Medical Image Classification](https://link.springer.com/chapter/10.1007/978-3-031-43987-2_10)* (MICCAI 2023)

## Contents

1. [Introduction](#1-introduction)
2. [Research and Data Exploration](#2-research-and-data-exploration): paper review, dataset choice, exploration, splits, image quality, enhancement
3. [Baseline Model](#3-baseline-model): training framework and a CNN trained from scratch
4. [Solution Improvement](#4-solution-improvement): transfer learning (3 pretrained models), class imbalance, image enhancement, GAN augmentation, plus a hyper-parameter search and a seed study
5. [Evaluation, Visualisation and Comparison](#5-evaluation-visualisation-and-comparison): test-set results, Grad-CAM, comparison with the baseline and with the paper
6. [Conclusion and Future Directions](#6-conclusion-and-future-directions)
7. [References](#references) and [Appendix](#appendix)

## 1. Introduction

**Problem.** Skin cancer is one of the most common cancers, and melanoma is its most dangerous form: caught early it is usually curable, caught late it is often fatal. A classifier for dermatoscopic images could act as a second reader and help prioritise patients.

**Task.** Classify a dermatoscopic image into one of 7 diagnostic categories (for example naevus, melanoma or basal cell carcinoma) on a highly imbalanced dataset, HAM10000.

**Objectives:**

| # | Objective | Where |
|---|---|---|
| 1 | Select and critically review a high-quality paper | §2.1 |
| 2 | Justify and describe the dataset (size, splits, classes, image quality, standardisation, enhancement) | §2.2–§2.10 |
| 3 | Build a CNN **baseline** that follows good practice, with every setting justified | §3 |
| 4 | **Improve** it: transfer learning (three pretrained models), imbalance handling, image enhancement, GAN images, a hyper-parameter search and a seed study | §4 |
| 5 | Evaluate with suitable metrics, **visualise** the results (learning curves, confusion matrices, Grad-CAM) and **compare** them with the baseline and the paper | §5 |
| 6 | Keep everything **reproducible** | whole notebook |

**Principles I followed throughout:**

1. **No data leakage.** HAM10000 often has several photos of one lesion, so I split the data by lesion, not by image. The paper's random image split is reproduced once to measure what it changes (§4.8, §5.6).
2. **Decisions use the validation set only.** The imbalance strategy, enhancement, hyper-parameters, early stopping and checkpoints never look at the test set, which is used only in §3.3 and §5.
3. **Accuracy is not enough.** 67 % of the images are one class, so always answering "naevus" is already 67 % accurate. The main metrics are **macro-F1** (the same family as the paper's F1), **balanced accuracy**, **melanoma recall** and one-vs-rest **AUC**. Per class I report recall, precision and F1.

### How to run this notebook

Open the notebook in VS Code, follow the set-up steps in the first code cell (§1.1) and press **Run All**; there is a single configuration.

* **Data:** if HAM10000 is not in `data/` yet, the first cells download it from Kaggle.
* **Saved results:** resized images and trained models are kept in `outputs/` and reloaded only if their settings and data are unchanged. Deleting `outputs/` forces a full re-run, which takes about an hour on the reference machine.
* **Reference machine:** AMD Ryzen 7 7800X3D, 64 GB RAM, AMD Radeon RX 7900 XT (20 GB), Windows 11, Python 3.12.10, PyTorch 2.9.1 (ROCm 7.2.1); speed benchmark in Appendix B.
* **Reproducibility:** every random generator uses seed 42 and the GPU kernels are deterministic. Another machine or driver can still give slightly different numbers, within the run-to-run noise measured in §4.6.

### 1.1 Set-up and configuration

All settings live in one cell, and each value is explained where it is used.

A few engineering choices make the notebook faster without changing what the models learn: a workaround for a BatchNorm bug in the Windows ROCm build of PyTorch (Appendix A); decoding the resized images once, caching them on disk and keeping them on the GPU, with augmentation on the GPU as well, because the CPU data loader was the bottleneck (Appendix B); 16-bit mixed precision, which was roughly two to five times faster than 32-bit in the benchmark; and saving trained models so that they are reused when their settings have not changed.

In [ ]:
#Set-up: create a virtual environment with Python 3.12 (e.g. `python -m venv venv`) and select it as the notebook kernel.
#Then, BEFORE running this cell, install PyTorch and torchvision built for your hardware:
#  NVIDIA, Linux:   `pip install torch torchvision` (the default build already supports CUDA).
#  NVIDIA, Windows: the default build is CPU-only; use the CUDA install command from https://pytorch.org/get-started/locally/
#  AMD, Linux:      use the ROCm install command from https://pytorch.org/get-started/locally/
#  AMD, Windows:    use AMD's ROCm build of PyTorch (see AMD's ROCm documentation); this is the set-up used here (Appendix A).
#  macOS:           `pip install torch torchvision`; current releases use the GPU only on Apple-silicon Macs (MPS), not NVIDIA or AMD.
#ROCm builds expose the GPU through torch.cuda, so NVIDIA and AMD GPUs are found the same way. Without a GPU the notebook still runs, but training takes far longer.
%pip install -q numpy pandas matplotlib opencv-python pillow scikit-learn tqdm kagglehub

In [ ]:
import os, platform, copy, hashlib, json, math, random, sys, time, warnings
from functools import partial
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

# Windows + AMD only: skip a GPU-detection probe that fails when the folder path contains spaces.
# Has to happen before torch is imported (Appendix A).
if os.name == "nt" and "ROCM_SDK_TARGET_FAMILY" not in os.environ:
    try:
        from rocm_sdk import _dist_info
        if "custom" in _dist_info.AVAILABLE_TARGET_FAMILIES:
            os.environ["ROCM_SDK_TARGET_FAMILY"] = "custom"
    except Exception:
        pass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import models as tvm

from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix, precision_recall_fscore_support)
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from tqdm import tqdm as _tqdm
from IPython.display import display

tqdm = partial(_tqdm, file=sys.stdout, ncols=100, mininterval=1)      # plain-text progress bars, saved with the notebook

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 30, "display.width", 140)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.autolayout": True})
print(f"Python {platform.python_version()} | torch {torch.__version__} | torchvision {torchvision.__version__} | "
      f"numpy {np.__version__} | pandas {pd.__version__} | opencv {cv2.__version__}")

In [ ]:
# Settings shared by all experiments
SEED = 42

# images
IMG_SIZE = 224                           # input size of the ImageNet models
IMAGENET_MEAN = (0.485, 0.456, 0.406)    # ImageNet statistics, expected by the pretrained weights
IMAGENET_STD = (0.229, 0.224, 0.225)
N_CLASSES = 7

# training
BATCH_SIZE = 64                          # throughput stops rising here on this GPU (Appendix B)
EPOCHS_CNN = 40                          # baseline; early stopping usually ends it sooner
EPOCHS_HEAD = 3                          # transfer learning stage 1: new output layer only
EPOCHS_FINETUNE = 25                     # transfer learning stage 2: whole network
PATIENCE = 5                             # early stopping, pretrained models
PATIENCE_CNN = 8                         # the baseline learns more slowly
N_SEARCH_TRIALS = 6
SEARCH_FT_EPOCHS = 3                     # kept short on purpose
GAN_STEPS = 5000                        # about 134 passes over the training images at batch 64; not tuned
N_BOOTSTRAP = 200                       # resamples for the 95 % confidence intervals

# reported by the paper on HAM10000 (Table 1b)
PAPER = {"name": "DiffMIC (Yang et al., MICCAI 2023)", "accuracy": 0.906, "f1": 0.816}

In [ ]:
# Download HAM10000 from Kaggle into data/ unless it is already there
import shutil, kagglehub
DATA_DIR = Path("data")
if not (DATA_DIR / "HAM10000_metadata.csv").exists():
    kaggle_dir = Path(kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000"))
    DATA_DIR.mkdir(exist_ok=True)
    for name in ["HAM10000_metadata.csv", "hmnist_8_8_L.csv", "hmnist_8_8_RGB.csv", "hmnist_28_28_L.csv", "hmnist_28_28_RGB.csv"]:
        shutil.copy2(kaggle_dir / name, DATA_DIR / name)
    for name in ["HAM10000_images_part_1", "HAM10000_images_part_2"]:
        shutil.copytree(kaggle_dir / name, DATA_DIR / name, dirs_exist_ok=True)
print("data jpg files:", len(list(DATA_DIR.rglob("*.jpg"))))

In [ ]:
DATA_DIR = Path("data")
OUT_DIR = Path("outputs/checkpoints")    # trained weights and training histories
CACHE_DIR = Path("outputs/cache")        # resized images (.npy)
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

META_CSV = DATA_DIR / "HAM10000_metadata.csv"
IMAGE_PATHS = {p.stem: p for p in DATA_DIR.rglob("*.jpg")}      # image_id -> file
print("metadata  :", META_CSV.resolve())
print("jpg files :", len(IMAGE_PATHS))

In [ ]:
def set_seed(seed):
    """Seed every random generator and make the GPU kernels deterministic."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


if torch.cuda.is_available():                      # AMD ROCm also shows up as "cuda"
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# Mixed precision (Appendix B). float16 needs a GradScaler; bfloat16 does not and is safer for transformers, so ViT uses it.
AMP_FP16 = torch.float16 if device.type == "cuda" else None
AMP_BF16 = torch.bfloat16 if device.type == "cuda" else None

set_seed(SEED)
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

In [ ]:
# Windows + AMD only (Appendix A): the ROCm build sends BatchNorm to MIOpen, whose run-time kernel compile fails with
# "'type_traits' file not found" (https://github.com/ROCm/TheRock/issues/9181). PyTorch's own GPU kernel does the same maths.
def apply_rocm_windows_batchnorm_fix():
    if os.name != "nt" or getattr(torch.version, "hip", None) is None:
        return False
    original = F.batch_norm

    def native_batch_norm(input, running_mean, running_var, weight=None, bias=None, training=False, momentum=0.1, eps=1e-5):
        if not input.is_cuda:
            return original(input, running_mean, running_var, weight, bias, training, momentum, eps)
        if training and input.numel() // max(input.shape[1], 1) <= 1:
            raise ValueError(f"Expected more than 1 value per channel when training, got {tuple(input.shape)}")
        previous = torch.backends.cudnn.enabled         # PyTorch 2.9 chooses the backend from this flag
        torch.backends.cudnn.enabled = False
        try:
            return torch.batch_norm(input, weight, bias, running_mean, running_var, training, momentum, eps, False)
        finally:
            torch.backends.cudnn.enabled = previous

    F.batch_norm = native_batch_norm
    return True


print("BatchNorm workaround for Windows + ROCm active:", apply_rocm_windows_batchnorm_fix())

In [ ]:
# Numbered captions: table captions go above the table, figure captions below the figure.
from IPython.display import Markdown

_CAPTION_COUNTER = {"Figure": 0, "Table": 0}


def caption(kind, text):
    _CAPTION_COUNTER[kind] += 1
    display(Markdown(f"**{kind} {_CAPTION_COUNTER[kind]}.** {text}"))


def show_fig(text):
    plt.show()
    caption("Figure", text)


def show_table(df, text):
    caption("Table", text)
    display(df)

## 2. Research and Data Exploration

### 2.1 Paper review

**Paper.** Yang, Y., Fu, H., Aviles-Rivero, A. I., Schönlieb, C.-B. and Zhu, L. (2023). *DiffMIC: Dual-Guidance Diffusion Network for Medical Image Classification.* MICCAI 2023, LNCS, pp. 95–105. DOI 10.1007/978-3-031-43987-2_10.

**Why this paper, and how I judged its quality.** MICCAI is a leading peer-reviewed medical-imaging conference. HAM10000 is one of the paper's three benchmarks, so its numbers are comparable with mine, and the code is public. The baselines are strong: ResNet, ViT, Swin, PVT and long-tail methods such as LDAM and ProCo. I judged quality by novelty, baseline strength, ablations and implementation detail, then read the paper critically for flaws in the evaluation.

**Method.** DiffMIC uses a diffusion model (DDPM) as a classifier: it adds noise to the label vector and learns to remove it, conditioned on the image, with a ResNet18 image embedding. Dual-granularity Conditional Guidance (DCG) adds a *global* prior from a saliency map of the whole image and a *local* prior from six 32×32 salient patches combined by gated attention. Both priors shift the noisy label and condition the denoiser, and a Maximum-Mean-Discrepancy loss (weight 0.5) pulls the predicted noise for each prior towards Gaussian noise. At inference the label is denoised from pure noise in T = 250 steps on HAM10000.

**Results.** On a 70/30 split of HAM10000, DiffMIC reports accuracy 0.906 and F1 0.816, against 0.887 and 0.763 for the best baseline, ProCo. The module ablation uses only the authors' placenta dataset, where diffusion, DCG and MMD each add roughly 1–3 accuracy points.

**Critical evaluation.**
1. *Possible leakage.* HAM10000 has 10,015 images of far fewer lesions, and the paper splits 7:3 without saying whether images of one lesion stay together, so its scores may be inflated. This is why I split by lesion.
2. *No variance.* Single runs without confidence intervals, so the gains over ProCo (0.019 accuracy, 0.053 F1) may be noise.
3. *Narrow metrics.* Only accuracy and F1 (averaging not stated); no sensitivity, AUC or melanoma recall.
4. *Weak support for some claims.* The ablation is not repeated on HAM10000, and the noise-removal claim rests on a t-SNE plot.
5. *Cost.* 1,000 epochs on an RTX 3090, and no validation protocol is described.

**What I take from it.** I use the same ResNet18 encoder, 224 px centre crop, flip and rotation augmentation and Adam (as AdamW), and report accuracy and macro-F1 for comparison. The data are split by lesion, and the paper's random split is run once to measure the leakage. Re-implementing the diffusion pipeline was beyond my compute and time, so §5.6 compares my results with the reported numbers and discusses the gap.

### 2.2 Why HAM10000

| Reason | Evidence |
|---|---|
| **Public, expert-labelled, citable** | 10,015 dermatoscopic images in 7 diagnostic classes; more than half of the diagnoses are confirmed by histopathology (Tschandl et al., 2018). |
| **Widely used benchmark** | It is the training set of ISIC 2018 Task 3 (Codella et al., 2019) and is used by the reviewed paper and the methods it compares with (DANIL, contrastive learning, ProCo), so results can be compared with the literature. |
| **Clinically important** | Melanoma detection is one of the best-known medical-imaging tasks (Esteva et al., 2017). |
| **Interesting problems** | strong class imbalance (about 58 : 1); a domain very different from ImageNet, which makes transfer learning a real question; artefacts such as hair, rulers and vignettes that motivate enhancement; and several images per lesion, which makes a leakage-free split necessary. |
| **Right size for the coursework** | about 2.8 GB and 10,000 images: every experiment trains in minutes on one GPU, and the data are still large enough for stable results. |

### 2.3 Dataset description

The metadata file has one row per image. The next cells load it and summarise the images.

In [ ]:
CLASSES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]            # alphabetical; this is the label order
CLASS_IDX = {c: i for i, c in enumerate(CLASSES)}
MEL = CLASS_IDX["mel"]
CLASS_INFO = {                                                          # code: (full name, category)
    "akiec": ("Actinic keratoses / intra-epithelial carcinoma", "pre-malignant"),
    "bcc":   ("Basal cell carcinoma", "malignant"),
    "bkl":   ("Benign keratosis-like lesions", "benign"),
    "df":    ("Dermatofibroma", "benign"),
    "mel":   ("Melanoma", "malignant"),
    "nv":    ("Melanocytic naevi", "benign"),
    "vasc":  ("Vascular lesions", "benign"),
}

meta = pd.read_csv(META_CSV)
meta["label"] = meta["dx"].map(CLASS_IDX)
meta["path"] = meta["image_id"].map(IMAGE_PATHS)
assert meta["path"].notna().all(), "some image files from the metadata were not found"

print(f"{len(meta)} images in the metadata")
show_table(meta.drop(columns=["path"]).head(), "First five rows of the metadata (one row per image).")

In [ ]:
# image sizes from the JPEG headers only, without decoding
def image_size(path):
    with Image.open(path) as im:
        return im.size                                   # (width, height)

with ThreadPoolExecutor(16) as pool:
    sizes = list(pool.map(image_size, meta["path"]))
size_counts = pd.Series(sizes).value_counts()

overview = pd.DataFrame({"value": {
    "images": len(meta),
    "distinct lesions (lesion_id)": meta["lesion_id"].nunique(),
    "images per lesion (mean)": round(len(meta) / meta["lesion_id"].nunique(), 2),
    "classes": meta["dx"].nunique(),
    "image size(s) in pixels (w x h)": ", ".join(f"{w}x{h} ({n})" for (w, h), n in size_counts.items()),
    "colour channels": "3 (RGB JPEG)",
    "images with missing age": int(meta["age"].isna().sum()),
    "total size on disk (GB)": round(sum(p.stat().st_size for p in meta["path"]) / 1e9, 2),
}})
show_table(overview, "Dataset overview: images, lesions, image size, channels, missing metadata and size on disk.")

class_table = (meta.groupby("dx").agg(images=("image_id", "count"), lesions=("lesion_id", "nunique"))
               .reindex(CLASSES))
class_table.insert(0, "category", [CLASS_INFO[c][1] for c in CLASSES])
class_table.insert(0, "diagnosis", [CLASS_INFO[c][0] for c in CLASSES])
class_table["share %"] = (100 * class_table["images"] / class_table["images"].sum()).round(1)
class_table.index.name = "class"
show_table(class_table.style.format({"share %": "{:.1f}"}).set_properties(subset=["diagnosis", "category"], **{"text-align": "left"}),
           "Images and lesions per diagnosis (the class code is the row label), with the clinical category and the share of the dataset.")

**Findings.** All images have the same format, 600 × 450 RGB JPEG. The 10,015 images show 7,470 distinct lesions, so some lesions appear more than once (Figure 1, right).

### 2.4 Class distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

counts = class_table["images"]
bars = axes[0].bar(CLASSES, counts, color=["#d95f02" if CLASS_INFO[c][1] != "benign" else "#1b9e77" for c in CLASSES])
for b, c in zip(bars, counts):
    axes[0].text(b.get_x() + b.get_width() / 2, c, f"{c}\n{100 * c / counts.sum():.1f}%", ha="center", va="bottom", fontsize=8)
axes[0].set_title("Images per class (orange = malignant / pre-malignant)")
axes[0].set_ylabel("number of images"); axes[0].set_ylim(0, counts.max() * 1.18)

axes[1].bar(CLASSES, counts, color="#7570b3"); axes[1].set_yscale("log")
axes[1].set_title("Same data, logarithmic axis"); axes[1].set_ylabel("number of images (log)")

per_lesion = meta.groupby("lesion_id").size()
axes[2].hist(per_lesion, bins=np.arange(1, per_lesion.max() + 2) - 0.5, color="#66a61e")
axes[2].set_yscale("log"); axes[2].set_xlabel("images of the same lesion"); axes[2].set_ylabel("number of lesions (log)")
axes[2].set_title("Several images of one lesion are common")
for ax in axes[:2]:
    ax.set_xlabel("diagnosis (class code, see the class table above)")
show_fig("Class distribution: images per class (left; orange = malignant or pre-malignant), the same on a log axis (middle) and how many images each lesion has (right).")

ratio = counts.max() / counts.min()
multi = (per_lesion > 1).sum()
print(f"imbalance ratio (largest / smallest class): {ratio:.1f} : 1")
print(f"majority-class accuracy (always predict 'nv'): {100 * counts.max() / counts.sum():.1f} %")
print(f"lesions photographed more than once: {multi} of {len(per_lesion)}  "
      f"({100 * meta['lesion_id'].map(per_lesion).gt(1).mean():.1f} % of all images belong to such lesions)")

**Findings.** `nv` is two thirds of the data, while `df` and `vasc` are only 1.1 % and 1.4 % (58 : 1). So accuracy alone is misleading (§1), the splits are stratified, and §4.2 compares a weighted loss with balanced sampling. Melanoma is the second-largest class (about 11 %), and I report it separately because a missed melanoma is the costliest error. 1,956 of the 7,470 lesions have more than one image, which is why the data are split by lesion (§2.6).

### 2.5 Patient and lesion metadata

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7.5))

axes[0, 0].hist(meta["age"].dropna(), bins=range(0, 95, 5), color="#4c72b0")
axes[0, 0].set_title(f"Age (years); {int(meta['age'].isna().sum())} missing"); axes[0, 0].set_xlabel("age")

sex = meta["sex"].fillna("unknown").value_counts()
axes[0, 1].bar(sex.index, sex.values, color="#55a868"); axes[0, 1].set_title("Sex")

loc = meta["localization"].value_counts()
axes[0, 2].barh(loc.index[::-1], loc.values[::-1], color="#c44e52"); axes[0, 2].set_title("Body location")

dxt = meta["dx_type"].value_counts()
axes[1, 0].bar(dxt.index, dxt.values, color="#8172b2"); axes[1, 0].set_title("How the diagnosis was confirmed")
axes[1, 0].tick_params(axis="x", rotation=20)

data = [meta.loc[meta["dx"] == c, "age"].dropna() for c in CLASSES]
axes[1, 1].boxplot(data, tick_labels=CLASSES, showfliers=False); axes[1, 1].set_title("Age by class")

tab = pd.crosstab(meta["dx"], meta["dx_type"]).reindex(CLASSES)
(tab.div(tab.sum(axis=1), axis=0) * 100).plot(kind="bar", stacked=True, ax=axes[1, 2], legend=True, width=0.8)
axes[1, 2].set_title("Confirmation type within each class (%)"); axes[1, 2].set_xlabel(""); axes[1, 2].tick_params(axis="x", rotation=0)
axes[1, 2].legend(fontsize=7)
show_fig("Metadata overview: age, sex, body location, how the diagnosis was confirmed, age per class and confirmation type per class.")

**Findings.** Like the paper, I treat this as pure image classification and do not feed the metadata to the models. Age differs by class (melanoma, carcinomas and keratoses are more common in older patients), which the models do not control for. Most lesions are on the back, lower extremity and trunk. Some diagnoses rest on follow-up or consensus rather than histopathology, so part of the labels are expert opinion. The 57 missing ages do not matter because age is not used.

### 2.6 Train / validation / test splits

**Every lesion goes to exactly one split.** `StratifiedGroupKFold` with `groups = lesion_id` and `y = diagnosis` keeps all images of a lesion together and the class proportions about equal; two successive splits give about **70 % train, 15 % validation and 15 % test**. Validation drives every decision and the test set is only used for reporting (§3.3, §5). For the leakage experiment (§4.8) I also build the paper's protocol, a random 70/30 split of *images*, holding out 10 % of its training part for validation.

In [ ]:
def lesion_level_split(df, seed):
    """70/15/15 split, stratified by diagnosis and grouped by lesion. Returns row positions."""
    outer = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=seed)          # 1/7 of the data -> test
    trainval, test = next(outer.split(df, df["dx"], groups=df["lesion_id"]))
    tv = df.iloc[trainval]
    inner = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=seed)          # 1/6 of the rest -> validation
    tr, va = next(inner.split(tv, tv["dx"], groups=tv["lesion_id"]))
    return trainval[tr], trainval[va], test


def paper_style_split(df, seed):
    """The paper's protocol: random 70/30 split of images (lesions ignored); 10 % of train held out for validation."""
    idx = np.arange(len(df))
    train, test = train_test_split(idx, test_size=0.30, stratify=df["dx"], random_state=seed)
    train, val = train_test_split(train, test_size=0.10, stratify=df["dx"].iloc[train], random_state=seed)
    return train, val, test


SPLITS = {}
for name, fn in [("lesion", lesion_level_split), ("paper", paper_style_split)]:
    tr, va, te = fn(meta, SEED)
    SPLITS[name] = {"train": np.sort(tr), "val": np.sort(va), "test": np.sort(te)}

# no lesion may appear in two parts of the lesion-level split
L = SPLITS["lesion"]
lesions = {k: set(meta["lesion_id"].iloc[v]) for k, v in L.items()}
assert not (lesions["train"] & lesions["val"]) and not (lesions["train"] & lesions["test"]) and not (lesions["val"] & lesions["test"])
print("lesion-level split: no lesion is shared between train, validation and test.\n")

# number of paper-style test images that show a lesion also in its training part
P = SPLITS["paper"]
train_lesions = set(meta["lesion_id"].iloc[P["train"]])
leaked = meta["lesion_id"].iloc[P["test"]].isin(train_lesions)
print(f"paper-style split: {leaked.sum()} of {len(P['test'])} test images ({100 * leaked.mean():.1f} %) show a lesion "
      f"that is also in the training set.\n")

rows = []
for proto in SPLITS:
    for part, idx in SPLITS[proto].items():
        c = meta["dx"].iloc[idx].value_counts().reindex(CLASSES).fillna(0).astype(int)
        rows.append({"protocol": proto, "split": part, "images": len(idx), "lesions": meta["lesion_id"].iloc[idx].nunique(),
                     **{k: v for k, v in c.items()}})
split_table = pd.DataFrame(rows)
show_table(split_table, "Images, lesions and class counts in the train, validation and test part of both split protocols.")

In [ ]:
# class proportions per split for both protocols, and the share of leaked test images
fig, axes = plt.subplots(1, 3, figsize=(16, 3.9), gridspec_kw={"width_ratios": [2.3, 2.3, 1.1]})
for ax, proto in zip(axes[:2], ["lesion", "paper"]):
    props = pd.DataFrame({part: meta["dx"].iloc[idx].value_counts(normalize=True).reindex(CLASSES).fillna(0) * 100
                          for part, idx in SPLITS[proto].items()})
    props.plot(kind="bar", ax=ax, width=0.8)
    ax.set_title(f"Class proportions per split ({proto} protocol)"); ax.set_ylabel("% of the split")
    ax.set_xlabel("diagnosis (class code)"); ax.tick_params(axis="x", rotation=0)

leak_share = {}
for proto in SPLITS:                                        # test images whose lesion is also in the training part
    train_l = set(meta["lesion_id"].iloc[SPLITS[proto]["train"]])
    leak_share[proto] = 100 * meta["lesion_id"].iloc[SPLITS[proto]["test"]].isin(train_l).mean()
axes[2].bar(["lesion\nprotocol", "paper\nprotocol"], [leak_share["lesion"], leak_share["paper"]], color=["#1b9e77", "#d95f02"])
for k, proto in enumerate(("lesion", "paper")):
    axes[2].text(k, leak_share[proto] + 1, f"{leak_share[proto]:.1f} %", ha="center")
axes[2].set_ylim(0, 50); axes[2].set_ylabel("% of test images with a lesion\nthat is also in the training set"); axes[2].set_title("Leakage into the test set")
show_fig("Left and middle: class proportions in the train, validation and test part of both split protocols (both are stratified, so the bars are almost equal by design). "
         f"Right: share of the test images whose lesion also appears in the training part ({leak_share['lesion']:.0f} % for the lesion-level protocol, {leak_share['paper']:.0f} % for the paper-style protocol).")

**Findings.** Both protocols are stratified, so the class mix is the same in every part and the left and middle panels look alike by design. The right panel shows the real difference: no test image of the lesion-level split shows a lesion from the training set, against 33 % (992 of 3,005) for the paper-style split. These near-duplicates are the leakage studied in §4.8 and §5.6. Everything below uses the lesion-level split unless stated otherwise.

### 2.7 Loading, centre-cropping and caching the images

Pretrained networks expect 224 × 224 input, and the images are 4:3. Like the paper, I **centre-crop the largest square (450 × 450) and resize it to 224 × 224** with Lanczos filtering. This keeps the lesion's proportions, and lesions sit near the centre anyway. Decoding 10,015 JPEGs in every epoch would take longer than the training itself, so the images are decoded **once**, saved as a `uint8` array (1.5 GB) and kept on the GPU (Appendix B).

In [ ]:
def load_square_224(path, size=IMG_SIZE):
    """Centre-crop the largest square of a JPEG and resize it; returns an RGB uint8 array."""
    with Image.open(path) as im:
        im = im.convert("RGB")
        w, h = im.size
        s = min(w, h)
        left, top = (w - s) // 2, (h - s) // 2
        return np.asarray(im.crop((left, top, left + s, top + s)).resize((size, size), Image.LANCZOS), dtype=np.uint8)


def cached_array(tag, ids, build_fn):
    """Load the uint8 array [N, H, W, 3] from CACHE_DIR, or build it with build_fn and save it."""
    key = hashlib.md5("|".join(ids).encode()).hexdigest()[:8]
    file = CACHE_DIR / f"cache_{tag}_{IMG_SIZE}_{len(ids)}_{key}.npy"
    if file.exists():
        print(f"[{tag}] loaded from {file.name}")
        return np.load(file)
    t0 = time.perf_counter()
    arr = build_fn()
    np.save(file, arr)
    print(f"[{tag}] built {arr.shape} in {time.perf_counter() - t0:.0f} s and saved to {file.name}")
    return arr


def _build_raw():
    with ThreadPoolExecutor(16) as pool:                                    # PIL releases the GIL while decoding
        return np.stack(list(tqdm(pool.map(load_square_224, meta["path"]), total=len(meta), desc="decoding images")))


IDS = meta["image_id"].tolist()
X_RAW = cached_array("raw", IDS, _build_raw)
Y = meta["label"].to_numpy()
print("raw cache:", X_RAW.shape, X_RAW.dtype, f"{X_RAW.nbytes / 1e9:.2f} GB")

### 2.8 Looking at the data and assessing image quality

Following the Week 6 advice to look at the data before training: five random images per class after cropping and resizing.

In [ ]:
def show_gallery(x, labels, per_class=5, seed=SEED, title=""):
    rng = np.random.RandomState(seed)
    fig, axes = plt.subplots(len(CLASSES), per_class, figsize=(2.0 * per_class, 2.05 * len(CLASSES)))
    for r, c in enumerate(CLASSES):
        pick = rng.choice(np.flatnonzero(labels == CLASS_IDX[c]), size=per_class, replace=False)
        for k in range(per_class):
            ax = axes[r, k]; ax.axis("off")
            if k < len(pick):
                ax.imshow(x[pick[k]])
            if k == 0:
                ax.set_title(f"{c}: {CLASS_INFO[c][0][:30]}", fontsize=8, loc="left")
    fig.suptitle(title, y=1.0); show_fig(title or "Random images of every class")


show_gallery(X_RAW, Y, title="Five random images per class (224 x 224, centre-cropped)")

**Observations.** Many images contain **hair** over the lesion; some have a dark circular vignette or are slightly blurred, and colour and lighting vary between devices and clinics. `mel`, `nv` and `bkl` often look alike.

Five **quality measures** per image (224 px version) put numbers on this: brightness (mean grey level), contrast (grey-level standard deviation), sharpness (variance of the Laplacian), the share of **dark corner** pixels (vignette) and the share of **hair** pixels (black-hat filter plus an elongation test).

In [ ]:
_HAIR_KERNEL = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))          # black-hat kernel for thin dark structures


def hair_mask(img_rgb, threshold=20, min_length=30, max_thickness=4.0):
    """Hair = thin dark structures (black-hat filter) that are also long and thin.

    A plain black-hat threshold also fires on pigment-network lines and dots inside the lesion (about 3 % of all
    pixels, see the check after the hair-removal figure), so a connected component is kept only if it is at least
    min_length pixels long and at most max_thickness pixels thick on average (area / length).
    """
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    dark = (cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, _HAIR_KERNEL) > threshold).astype(np.uint8)
    _, comp, stats, _ = cv2.connectedComponentsWithStats(dark, connectivity=8)
    length = np.maximum(stats[:, cv2.CC_STAT_WIDTH], stats[:, cv2.CC_STAT_HEIGHT])
    keep = (length >= min_length) & (stats[:, cv2.CC_STAT_AREA] / np.maximum(length, 1) <= max_thickness)
    keep[0] = False                                                     # label 0 is the background
    return keep[comp].astype(np.uint8)


def quality_features(img):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    k = 40                                                                  # 40 x 40 corner squares
    corners = np.concatenate([gray[:k, :k].ravel(), gray[:k, -k:].ravel(), gray[-k:, :k].ravel(), gray[-k:, -k:].ravel()])
    return (gray.mean(), gray.std(), cv2.Laplacian(gray, cv2.CV_64F).var(), (corners < 25).mean(), hair_mask(img).mean())


quality = pd.DataFrame([quality_features(im) for im in tqdm(X_RAW, desc="quality")],
                       columns=["brightness", "contrast", "sharpness", "dark_corner_frac", "hair_frac"])
quality["dx"] = meta["dx"].values
q_summary = quality.groupby("dx").mean(numeric_only=True).reindex(CLASSES)
q_summary[["dark_corner_frac", "hair_frac"]] *= 100                                  # as % of the pixels
q_summary = q_summary.rename(columns={"dark_corner_frac": "dark corner %", "hair_frac": "hair %"})
q_summary.index.name = "class"
show_table(q_summary.round(2), "Mean image-quality measures per diagnosis (dark corners and hair as % of the pixels).")

fig, axes = plt.subplots(1, 5, figsize=(17, 3.2))
for ax, col in zip(axes, ["brightness", "contrast", "sharpness", "dark_corner_frac", "hair_frac"]):
    # sharpness spans several orders of magnitude, so it gets log-spaced bins
    bins = np.logspace(np.log10(max(quality[col].min(), 1e-3)), np.log10(quality[col].max()), 40) if col == "sharpness" else 40
    ax.hist(quality[col], bins=bins, color="#4c72b0"); ax.set_title(col)
    ax.set_xlabel(col + (" (log scale)" if col == "sharpness" else ""))
axes[2].set_xscale("log"); axes[0].set_ylabel("number of images")
show_fig("Distribution of the five image-quality measures over all images (sharpness on a log axis).")

In [ ]:
# the worst examples of each problem, then how common it is (rough thresholds)
def show_examples(order, title, n=6):
    fig, axes = plt.subplots(1, n, figsize=(2.2 * n, 2.5))
    for ax, i in zip(axes, order[:n]):
        ax.imshow(X_RAW[i]); ax.axis("off"); ax.set_title(f"{meta['dx'][i]}", fontsize=8)
    fig.suptitle(title, y=1.02); show_fig(title)


show_examples(np.argsort(quality["sharpness"].values), "Blurriest images (lowest Laplacian variance)")
show_examples(np.argsort(-quality["hair_frac"].values), "Most hair (highest black-hat fraction)")
show_examples(np.argsort(-quality["dark_corner_frac"].values), "Strongest dark vignette")
for t in (0.005, 0.01, 0.03):
    print(f"share of images whose hair fraction exceeds {100 * t:.1f} %: {100 * (quality['hair_frac'] > t).mean():.1f} %")
print(f"share of images with a dark vignette (> 20 % dark corner pixels): {100 * (quality['dark_corner_frac'] > 0.2).mean():.1f} %")
print(f"share of images with very low sharpness (Laplacian variance < 20): {100 * (quality['sharpness'] < 20).mean():.1f} %")

**Quality assessment and decisions.**

* **Size and format** are identical for every image; none is rejected.
* **Sharpness, brightness and contrast** vary between clinics, but no image is unusable, so I **keep all images**. Removing hard cases would make the test set unrealistically easy, and the brightness and contrast jitter of the augmentation (§3.1) covers the variation.
* **Hair**: 14.1 % of the images have more than 0.5 % hair pixels. The measure is a proxy (long, thin dark structures) that can also catch thin dark lesion borders. Hair removal is tested in §2.9 and §4.3.
* **Dark vignettes** almost disappear after the centre crop (0.2 % of the images).
* **The quality measures differ between classes** (Table 5): mean contrast is 20-22 for `akiec`, `bcc` and `df` but 28-34 for `nv`, `mel` and `vasc`, and mean sharpness ranges from 109 (`vasc`) to 203 (`bcc`). A model could exploit this bias; Grad-CAM (§5.4) checks where it looks.

### 2.9 Image enhancement

Two classical techniques from Week 1, applied once and cached:

1. **Hair removal**, similar to *DullRazor* (Lee et al., 1997): a black-hat filter finds thin dark structures, an elongation test keeps the long thin ones, and the dilated mask is filled by **inpainting** (Telea, OpenCV).
2. **CLAHE** (Zuiderveld, 1994) on the LAB lightness channel: it raises *local* contrast without amplifying noise as global histogram equalisation (Lab 5) does, and leaves the colours unchanged.

In [ ]:
def remove_hair(img_rgb, threshold=20):
    """Black-hat hair mask, dilated, then inpainted. Returns (clean image, mask)."""
    mask = hair_mask(img_rgb, threshold)
    mask = cv2.dilate(mask, np.ones((3, 3), np.uint8))
    return cv2.inpaint(img_rgb, mask * 255, 3, cv2.INPAINT_TELEA), mask


def clahe_lightness(img_rgb, clip=2.0, tiles=8):
    """CLAHE on the L channel of LAB (a new CLAHE object per call, it is not thread-safe)."""
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    lab[..., 0] = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tiles, tiles)).apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)


def enhance(img_rgb):
    return clahe_lightness(remove_hair(img_rgb)[0])


# visual check on the six hairiest images
hairy = np.argsort(-quality["hair_frac"].values)[:6]
fig, axes = plt.subplots(4, 6, figsize=(13, 9))
for k, i in enumerate(hairy):
    img = X_RAW[i]; clean, mask = remove_hair(img)
    for r, (im, ttl) in enumerate([(img, "original"), (mask * 255, "detected hair"), (clean, "hair removed"), (clahe_lightness(clean), "+ CLAHE")]):
        axes[r, k].imshow(im, cmap="gray" if r == 1 else None); axes[r, k].axis("off")
        if k == 0:
            axes[r, k].set_title(ttl, loc="left", fontsize=9)
show_fig("Hair removal and CLAHE on the six images with the most hair-like structures: original, detected hair, hair removed, hair removed plus CLAHE.")

# share of pixels flagged by a plain black-hat threshold and by the final (dilated) mask, on 400 random images
sample = np.random.RandomState(1).choice(len(X_RAW), 400, replace=False)
plain_share = np.mean([(cv2.morphologyEx(cv2.cvtColor(X_RAW[i], cv2.COLOR_RGB2GRAY), cv2.MORPH_BLACKHAT, _HAIR_KERNEL) > 20).mean() for i in sample])
mask_share = np.mean([cv2.dilate(hair_mask(X_RAW[i]), np.ones((3, 3), np.uint8)).mean() for i in sample])
print(f"share of pixels flagged on 400 random images: plain black-hat threshold {100 * plain_share:.1f} %, hair mask after the elongation test and dilation {100 * mask_share:.2f} %")

In [ ]:
def _build_enh():
    with ThreadPoolExecutor(16) as pool:                                    # OpenCV releases the GIL too
        return np.stack(list(tqdm(pool.map(enhance, X_RAW), total=len(X_RAW), desc="enhancing")))


X_ENH = cached_array("enh2", IDS, _build_enh)     # "enh2": built after the hair filter was made stricter
print("enhanced cache:", X_ENH.shape)

**Findings.** On very hairy images the mask follows the thick hairs and inpainting removes them, leaving some smeared patches. Thin or light hairs stay, and thin dark lesion borders can still be caught. A plain black-hat threshold flags 2.7 % of the pixels on 400 random images and also fires inside the lesion; the elongation test brings this down to 0.58 % after dilation (about 0.2 % before dilation, Table 5). Whether this helps the classifier is tested in §4.3.

### 2.10 Standardisation

Pixel values are scaled to 0-1 and normalised per channel with the **ImageNet** statistics that the pretrained weights expect. The baseline uses the same values, so every model gets identical input. The next cell compares them with the statistics of the training images.

In [ ]:
# per-channel mean and std of the training pixels, in chunks (one float64 copy would need about 8 GB)
tr_rows = SPLITS["lesion"]["train"]
s1, s2, n_px = np.zeros(3), np.zeros(3), 0
for a in range(0, len(tr_rows), 512):
    chunk = X_RAW[tr_rows[a:a + 512]].reshape(-1, 3).astype(np.float64) / 255.0
    s1 += chunk.sum(0); s2 += (chunk ** 2).sum(0); n_px += len(chunk)
ds_mean = s1 / n_px
ds_std = np.sqrt(s2 / n_px - ds_mean ** 2)
show_table(pd.DataFrame({"HAM10000 train mean": ds_mean, "ImageNet mean": IMAGENET_MEAN,
                      "HAM10000 train std": ds_std, "ImageNet std": IMAGENET_STD}, index=list("RGB")).round(3),
           "Per-channel mean and standard deviation of the training pixels compared with the ImageNet statistics.")

**Findings.** The training images are much *redder* than ImageNet (red mean 0.76 against 0.49), brighter and less varied. I still use the **ImageNet values**, because the pretrained first-layer filters were learned on inputs normalised this way, and all models get the same scaling. In ResNet18 and EfficientNet-B0 the batch-norm layers also adapt their statistics during fine-tuning. I did not test dataset-specific statistics. Augmentation and normalisation run **on the GPU** inside the loader (§3.1); validation and test images are only normalised.

#### Summary of the data-preparation decisions

| Step | Decision | Justification |
|---|---|---|
| Split | by lesion, stratified, 70/15/15 | no leakage, the same class mix in every part, a separate validation set |
| Spatial | centre crop 450 × 450, resized to 224 × 224 | as in the paper; keeps the aspect ratio; fits the pretrained models |
| Quality | keep all images | realistic test set; variation covered by augmentation |
| Enhancement | hair removal + CLAHE, tested in §4.3 | decided by the validation result |
| Standardisation | ÷255, then ImageNet mean/std | what the pretrained weights expect; the same for all models |
| Storage | `uint8` cache on the GPU | removes the CPU decoding bottleneck (Appendix B) |

## 3. Baseline Model

### 3.1 Training framework

All experiments in §3 and §4 share one small framework, so they differ **only** in the thing being tested:

| Component | What it does |
|---|---|
| `GpuLoader` | serves batches straight from the `uint8` image cache on the GPU; can draw all classes equally often and add extra (synthetic) training images |
| `gpu_augment` | random flips, rotation, zoom, shift, brightness and contrast for a whole batch on the GPU |
| `build_model` | the baseline CNN, or a pretrained torchvision model with a new 7-class output layer |
| `train_model` | staged training with AdamW, learning-rate scheduling, early stopping on validation **macro-F1**, best-checkpoint restore, saving and loading |
| `compute_metrics` | accuracy, balanced accuracy, macro-F1, AUC, per-class recall, precision, F1 and specificity, melanoma recall |

**Augmentation.** A dermatoscopic image has **no fixed orientation** (unlike a chest X-ray, Week 6), so flips and rotations keep the label. Small zooms and shifts imitate framing differences, and ±20 % brightness and contrast jitter imitates devices and lighting; I leave hue and saturation alone because colour is diagnostic. Only **training images** are augmented.

In [ ]:
def to_device_u8(arr):
    """uint8 [N, H, W, 3] array -> uint8 [N, 3, H, W] tensor on the device."""
    return torch.from_numpy(arr).permute(0, 3, 1, 2).contiguous().to(device)


MEAN_T = torch.tensor(IMAGENET_MEAN, device=device).view(1, 3, 1, 1)
STD_T = torch.tensor(IMAGENET_STD, device=device).view(1, 3, 1, 1)


def normalise(x01):
    return (x01 - MEAN_T) / STD_T


# augmentation strength; "full" is the default, §4.3b compares all three
AUG_PRESETS = {
    "full": dict(rotation=180, flips=True, zoom=0.10, shift=0.05, brightness=0.20, contrast=0.20),
    "mild": dict(rotation=15, flips=False, zoom=0.05, shift=0.03, brightness=0.10, contrast=0.10),   # cautious, like the X-ray example in Week 6
    "none": None,
}
DEFAULT_AUG = "full"             # §4.3b may change it for the later experiments


def gpu_augment(x, preset="full"):
    """Random augmentation of a whole batch on the GPU. x: float [B, 3, H, W] in [0, 1]."""
    cfg = AUG_PRESETS[preset]
    if cfg is None:
        return x
    b, dev = x.shape[0], x.device
    u = lambda: torch.rand(b, device=dev) * 2 - 1                        # one value in [-1, 1] per image
    angle = u() * math.radians(cfg["rotation"])
    scale = 1.0 + cfg["zoom"] * u()
    shift_x, shift_y = cfg["shift"] * u(), cfg["shift"] * u()
    cos, sin = angle.cos() * scale, angle.sin() * scale
    # rotation, zoom and shift as one affine matrix per image
    theta = torch.stack([torch.stack([cos, -sin, shift_x], 1), torch.stack([sin, cos, shift_y], 1)], 1)
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="reflection", align_corners=False)
    if cfg["flips"]:
        x = torch.where(torch.rand(b, 1, 1, 1, device=dev) < 0.5, x.flip(3), x)      # horizontal
        x = torch.where(torch.rand(b, 1, 1, 1, device=dev) < 0.5, x.flip(2), x)      # vertical
    x = x * (1 + cfg["brightness"] * u().view(b, 1, 1, 1))
    mean = x.mean(dim=(1, 2, 3), keepdim=True)
    x = (x - mean) * (1 + cfg["contrast"] * u().view(b, 1, 1, 1)) + mean             # contrast around the image mean
    return x.clamp(0, 1)


class GpuLoader:
    """Batches from a uint8 image tensor that is already on the GPU.

    train=True shuffles, augments and drops the last incomplete batch (BatchNorm needs more than one sample).
    balanced=True draws every class equally often. extra=(xe, ye) adds synthetic training images.
    """

    def __init__(self, x, y, idx, batch_size, train=False, balanced=False, extra=None, aug="full"):
        self.x, self.y, self.bs, self.train, self.balanced, self.aug = x, y, batch_size, train, balanced, aug
        self.idx = torch.as_tensor(idx, device=x.device, dtype=torch.long)
        self.xe, self.ye = extra if extra is not None else (None, None)
        self.n_main = len(self.idx)
        self.n_extra = 0 if self.xe is None else len(self.ye)
        if balanced:                                           # weight of an image = 1 / size of its class
            labels = torch.cat([y[self.idx], self.ye]) if self.n_extra else y[self.idx]
            self.w = (1.0 / torch.bincount(labels, minlength=N_CLASSES).clamp(min=1).float())[labels]

    def __len__(self):
        n = self.n_main + self.n_extra
        return n // self.bs if self.train else math.ceil(n / self.bs)

    def _gather(self, order):
        # positions below n_main are real images, the rest are synthetic
        if not self.n_extra:
            sel = self.idx[order]
            return self.x[sel], self.y[sel]
        real = order < self.n_main
        sel, extra_sel = self.idx[order[real]], order[~real] - self.n_main
        return torch.cat([self.x[sel], self.xe[extra_sel]]), torch.cat([self.y[sel], self.ye[extra_sel]])

    def __iter__(self):
        n = self.n_main + self.n_extra
        if not self.train:
            order = torch.arange(n, device=self.x.device)
        elif self.balanced:
            order = torch.multinomial(self.w, n, replacement=True)
        else:
            order = torch.randperm(n, device=self.x.device)
        for b in range(len(self)):
            xb, yb = self._gather(order[b * self.bs:(b + 1) * self.bs])
            xb = xb.float().div_(255)
            if self.train:
                xb = gpu_augment(xb, self.aug)
            yield normalise(xb), yb


X_RAW_T, X_ENH_T = to_device_u8(X_RAW), to_device_u8(X_ENH)
Y_T = torch.as_tensor(Y, device=device, dtype=torch.long)
print(f"image tensors on {device}: raw {tuple(X_RAW_T.shape)}, enhanced {tuple(X_ENH_T.shape)}, "
      f"{(X_RAW_T.numel() + X_ENH_T.numel()) / 1e9:.2f} GB")

In [ ]:
# eight random augmentations of one melanoma image
set_seed(SEED)
i = int(np.flatnonzero(Y == MEL)[0])
batch = X_RAW_T[i:i + 1].repeat(8, 1, 1, 1).float() / 255
aug = gpu_augment(batch)
fig, axes = plt.subplots(1, 9, figsize=(17, 2.2))
axes[0].imshow(batch[0].permute(1, 2, 0).cpu()); axes[0].set_title("original", fontsize=8)
for k in range(8):
    axes[k + 1].imshow(aug[k].permute(1, 2, 0).cpu()); axes[k + 1].set_title(f"augmented {k + 1}", fontsize=8)
for ax in axes: ax.axis("off")
show_fig("One melanoma image and eight random augmentations of it (training-time augmentation).")

In [ ]:
def compute_metrics(y, prob):
    """All metrics of this notebook from the true labels [N] and the predicted probabilities [N, 7]."""
    prob = prob.astype(np.float64)
    prob = prob / prob.sum(1, keepdims=True)                   # roc_auc_score checks that rows sum to 1
    pred = prob.argmax(1)
    precision, recall, f1, _ = precision_recall_fscore_support(y, pred, labels=range(N_CLASSES), zero_division=0)
    cm = confusion_matrix(y, pred, labels=range(N_CLASSES))
    tp = np.diag(cm)
    fp, fn = cm.sum(0) - tp, cm.sum(1) - tp
    tn = cm.sum() - tp - fp - fn
    specificity = tn / np.maximum(tn + fp, 1)                  # one-vs-rest
    return {"accuracy": accuracy_score(y, pred), "balanced_accuracy": balanced_accuracy_score(y, pred),
            "macro_f1": f1_score(y, pred, average="macro", zero_division=0),
            "auc_macro": roc_auc_score(y, prob, multi_class="ovr", average="macro", labels=list(range(N_CLASSES))),
            "mel_recall": recall[MEL], "mel_specificity": specificity[MEL],
            "recall": recall, "precision": precision, "f1": f1, "specificity": specificity}


def bootstrap_ci(y, pred, metric, n=N_BOOTSTRAP, seed=SEED):
    """95 % percentile bootstrap interval of metric(y_true, y_pred), resampling the test images."""
    rng = np.random.RandomState(seed)
    values = [metric(y[i], pred[i]) for i in (rng.randint(0, len(y), len(y)) for _ in range(n))]
    return np.percentile(values, [2.5, 97.5])


def inverse_frequency_weights(labels):
    """Class weights N / (K * n_c), as in Lab 5."""
    counts = np.bincount(labels, minlength=N_CLASSES).clip(min=1)
    return torch.tensor(len(labels) / (N_CLASSES * counts), dtype=torch.float32, device=device)

In [ ]:
class SimpleCNN(nn.Module):
    """Baseline CNN: 5 x [conv 3x3, BatchNorm, ReLU, max-pool], global average pooling, dropout, linear."""

    def __init__(self, num_classes=N_CLASSES, channels=(32, 64, 128, 256, 256), dropout=0.3):
        super().__init__()
        layers, prev = [], 3
        for c in channels:
            layers += [nn.Conv2d(prev, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True), nn.MaxPool2d(2)]
            prev = c
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(dropout), nn.Linear(prev, num_classes))

    def forward(self, x):
        return self.head(self.features(x))


def build_model(arch, pretrained=True, num_classes=N_CLASSES):
    """The model with a new num_classes output layer; pretrained weights are torchvision's ImageNet-1k."""
    if arch == "simple_cnn":
        return SimpleCNN(num_classes)
    if arch == "resnet18":
        m = tvm.resnet18(weights=tvm.ResNet18_Weights.DEFAULT if pretrained else None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif arch == "efficientnet_b0":
        m = tvm.efficientnet_b0(weights=tvm.EfficientNet_B0_Weights.DEFAULT if pretrained else None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    elif arch == "vit_b_16":
        m = tvm.vit_b_16(weights=tvm.ViT_B_16_Weights.DEFAULT if pretrained else None)
        m.heads.head = nn.Linear(m.heads.head.in_features, num_classes)
    else:
        raise ValueError(arch)
    return m


def head_module(model, arch):
    """The new classifier, which transfer learning trains first."""
    attribute = {"simple_cnn": "head", "resnet18": "fc", "efficientnet_b0": "classifier", "vit_b_16": "heads"}[arch]
    return getattr(model, attribute)


net = build_model("simple_cnn")
n_params = sum(p.numel() for p in net.parameters())
print(f"SimpleCNN: {n_params / 1e6:.2f} M parameters")
with torch.no_grad():                                                         # feature-map size after each block
    x = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)
    for layer in net.features:
        x = layer(x)
        if isinstance(layer, nn.MaxPool2d):
            print(f"  after pooling block: {tuple(x.shape)}")
del net

In [ ]:
@torch.no_grad()
def predict(model, loader, amp_dtype=None):
    """Mean loss, probabilities and labels over a loader (eval mode, no augmentation)."""
    model.eval()
    probs, ys, loss_sum, n = [], [], 0.0, 0
    for xb, yb in loader:
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=amp_dtype is not None):
            logits = model(xb)
        logits = logits.float()
        loss_sum += F.cross_entropy(logits, yb, reduction="sum").item()
        n += len(yb)
        probs.append(logits.softmax(1).cpu())
        ys.append(yb.cpu())
    return loss_sum / n, torch.cat(probs).numpy(), torch.cat(ys).numpy()


def train_one_epoch(model, loader, criterion, optimizer, scaler, amp_dtype, clip=1.0):
    model.train()
    loss_sum, correct, n = 0.0, 0, 0
    for xb, yb in loader:
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=amp_dtype is not None):
            logits = model(xb)
        loss = criterion(logits.float(), yb)
        if scaler.is_enabled():                     # float16: scale the loss so small gradients don't underflow
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip)
            optimizer.step()
        loss_sum += loss.item() * len(yb)
        correct += (logits.argmax(1) == yb).sum().item()
        n += len(yb)
    return loss_sum / n, correct / n


def fit_stage(model, train_loader, val_loader, *, epochs, param_groups, weight_decay, criterion, amp_dtype,
              patience, stage_name, history, verbose=True):
    """One training stage: AdamW + ReduceLROnPlateau, early stopping on validation macro-F1, best epoch restored."""
    optimizer = torch.optim.AdamW(param_groups, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.3, patience=2)
    scaler = torch.amp.GradScaler("cuda", enabled=(amp_dtype == torch.float16 and device.type == "cuda"))
    best_f1, best_state, best_epoch, bad_epochs = -1.0, None, 0, 0
    for epoch in range(1, epochs + 1):
        t0 = time.perf_counter()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler, amp_dtype)
        val_loss, prob, y = predict(model, val_loader, amp_dtype)
        m = compute_metrics(y, prob)
        scheduler.step(m["macro_f1"])
        history.append({"stage": stage_name, "epoch": len(history) + 1, "train_loss": train_loss, "train_acc": train_acc,
                        "val_loss": val_loss, "val_acc": m["accuracy"], "val_macro_f1": m["macro_f1"],
                        "val_bal_acc": m["balanced_accuracy"], "lr": optimizer.param_groups[0]["lr"],
                        "seconds": time.perf_counter() - t0})
        if verbose:
            print(f"  [{stage_name}] epoch {epoch:2d}/{epochs}  train loss {train_loss:.3f} acc {train_acc:.3f} | "
                  f"val loss {val_loss:.3f} acc {m['accuracy']:.3f} macro-F1 {m['macro_f1']:.3f} | "
                  f"lr {optimizer.param_groups[0]['lr']:.1e} | {time.perf_counter() - t0:.1f}s")
        if m["macro_f1"] > best_f1 + 1e-4:
            best_f1, best_epoch, bad_epochs = m["macro_f1"], epoch, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                if verbose:
                    print(f"  [{stage_name}] early stop (no improvement for {patience} epochs); best epoch {best_epoch}")
                break
    model.load_state_dict(best_state)                # back to the best epoch
    return best_f1

In [ ]:
RUNS = {}          # experiment name -> model, history, settings, training minutes


def data_fingerprint(data):
    """Short id of the images and the training split, so that outdated saved models are retrained."""
    split = hashlib.md5(np.ascontiguousarray(data["idx"]["train"]).tobytes()).hexdigest()[:6]
    return f"{split}-{len(data['x'])}-{float(data['x'][:16].float().mean()):.3f}"


def train_model(name, arch, data, *, pretrained, stages, class_weights=None, balanced_sampler=False, amp_dtype=None,
                extra=None, seed=SEED, weight_decay=1e-4, batch_size=BATCH_SIZE, patience=PATIENCE, verbose=True, aug=None):
    """Train one experiment, or load it if a model with exactly these settings was saved before.

    stages: list of {"name", "epochs", "lr", "train": "head" | "all", "head_mult"}. With "all" the whole network
            is fine-tuned, the pretrained layers at lr and the new layer at lr * head_mult.
    data:   {"x": uint8 images, "y": labels, "idx": {"train", "val", "test"}}
    """
    aug = aug or DEFAULT_AUG
    extra_id = 0 if extra is None else f"{len(extra[1])}-{hashlib.md5(extra[0][:8].cpu().numpy().tobytes()).hexdigest()[:6]}"
    config = {"arch": arch, "pretrained": pretrained, "stages": stages, "class_weights": class_weights is not None,
              "balanced_sampler": balanced_sampler, "extra_images": extra_id, "seed": seed, "weight_decay": weight_decay,
              "batch_size": batch_size, "patience": patience, "amp": str(amp_dtype), "data": data_fingerprint(data)}
    if aug != "full":                                     # only stored when it differs from the default
        config["aug"] = aug
    signature = hashlib.md5(json.dumps(config, sort_keys=True, default=str).encode()).hexdigest()[:10]
    ckpt, hist_file = OUT_DIR / f"ckpt_{name}.pt", OUT_DIR / f"hist_{name}.json"

    set_seed(seed)                                        # the new layer's initial weights depend on it
    model = build_model(arch, pretrained).to(device)

    if ckpt.exists() and hist_file.exists():
        saved = json.loads(hist_file.read_text())
        if saved.get("signature") == signature:
            model.load_state_dict(torch.load(ckpt, map_location=device))
            RUNS[name] = {"model": model, "history": saved["history"], "config": saved["config"], "minutes": saved["minutes"],
                          "from_cache": True, "arch": arch, "amp": amp_dtype}
            print(f"[{name}] loaded saved model")
            return RUNS[name]
        print(f"[{name}] settings or data changed since it was saved -> retraining")

    t0 = time.perf_counter()
    train_loader = GpuLoader(data["x"], data["y"], data["idx"]["train"], batch_size, train=True, balanced=balanced_sampler, extra=extra, aug=aug)
    val_loader = GpuLoader(data["x"], data["y"], data["idx"]["val"], 256)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    history, head = [], head_module(model, arch)
    head_ids = {id(p) for p in head.parameters()}
    for st in stages:
        if st["train"] == "head":
            for p in model.parameters():
                p.requires_grad = id(p) in head_ids
            groups = [{"params": list(head.parameters()), "lr": st["lr"]}]
        else:
            for p in model.parameters():
                p.requires_grad = True
            backbone = [p for p in model.parameters() if id(p) not in head_ids]
            groups = [{"params": backbone, "lr": st["lr"]}, {"params": list(head.parameters()), "lr": st["lr"] * st.get("head_mult", 10)}]
        if verbose:
            n_trainable = sum(p.numel() for g in groups for p in g["params"])
            n_all = sum(p.numel() for p in model.parameters())
            print(f"[{name}] stage '{st['name']}': {n_trainable:,} of {n_all:,} parameters trainable ({100 * n_trainable / n_all:.2f} %), lr {st['lr']:g}")
        fit_stage(model, train_loader, val_loader, epochs=st["epochs"], param_groups=groups, weight_decay=weight_decay,
                  criterion=criterion, amp_dtype=amp_dtype, patience=patience, stage_name=st["name"], history=history, verbose=verbose)
    minutes = (time.perf_counter() - t0) / 60
    torch.save(model.state_dict(), ckpt)
    hist_file.write_text(json.dumps({"history": history, "config": config, "minutes": minutes, "signature": signature}))
    RUNS[name] = {"model": model, "history": history, "config": config, "minutes": minutes, "from_cache": False,
                  "arch": arch, "amp": amp_dtype}
    print(f"[{name}] done in {minutes:.1f} min")
    return RUNS[name]


PRED_CACHE = {}


def evaluate(name, data, split):
    """Metrics, probabilities, labels and loss of a trained run on 'train', 'val' or 'test' (computed once)."""
    key = (name, split, id(data))
    if key not in PRED_CACHE:
        run = RUNS[name]
        loss, prob, y = predict(run["model"], GpuLoader(data["x"], data["y"], data["idx"][split], 256), run["amp"])
        PRED_CACHE[key] = (compute_metrics(y, prob), prob, y, loss)
    return PRED_CACHE[key]


def summary_row(name, m):
    return {"model": name, "accuracy": m["accuracy"], "balanced_acc": m["balanced_accuracy"], "macro_f1": m["macro_f1"],
            "auc": m["auc_macro"], "mel_recall": m["mel_recall"], "mel_spec": m["mel_specificity"]}


def per_class_table(names, data, split="val", labels=None, metrics=("recall", "precision", "f1")):
    """Per-class metrics of several runs: one row per class, one column per (metric, run).
    data is one data dict for all runs, or a list with one per run."""
    datas = data if isinstance(data, list) else [data] * len(names)
    labels = labels or names
    table = pd.DataFrame({(metric, label): evaluate(n, d, split)[0][metric]
                          for metric in metrics for n, d, label in zip(names, datas, labels)}, index=CLASSES)
    if all(d["idx"] is datas[0]["idx"] for d in datas):            # same images for every run
        table[("images", "")] = np.bincount(evaluate(names[0], datas[0], split)[2], minlength=N_CLASSES)
    return table


def plot_histories(names, title="", figsize=(14, 4)):
    """Loss, validation macro-F1 and accuracy curves of several runs."""
    fig, axes = plt.subplots(1, 3, figsize=figsize)
    for n in names:
        h = pd.DataFrame(RUNS[n]["history"])
        line, = axes[0].plot(h["epoch"], h["train_loss"], label=n)
        axes[0].plot(h["epoch"], h["val_loss"], "--", color=line.get_color())
        axes[1].plot(h["epoch"], h["val_macro_f1"], color=line.get_color(), label=n)
        axes[2].plot(h["epoch"], h["train_acc"], color=line.get_color(), label=f"{n} train")
        axes[2].plot(h["epoch"], h["val_acc"], "--", color=line.get_color())
    axes[0].set_title("loss (solid: train, dashed: validation)")
    axes[1].set_title("validation macro-F1")
    axes[2].set_title("accuracy (solid: train, dashed: validation)")
    for ax in axes:
        ax.set_xlabel("epoch")
    axes[0].legend(fontsize=7)
    fig.suptitle(title)
    show_fig(title or "Learning curves")


def plot_confusion(y, prob, title="", ax=None):
    """Row-normalised confusion matrix with counts (the diagonal is the recall of each class)."""
    cm = confusion_matrix(y, prob.argmax(1), labels=range(N_CLASSES))
    norm = cm / cm.sum(1, keepdims=True).clip(min=1)
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(5.2, 4.6))
    ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
    ax.grid(False)
    ax.set_xticks(range(N_CLASSES), CLASSES, rotation=45)
    ax.set_yticks(range(N_CLASSES), CLASSES)
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            ax.text(j, i, f"{100 * norm[i, j]:.0f}%\n({cm[i, j]})", ha="center", va="center", fontsize=6.5,
                    color="white" if norm[i, j] > 0.5 else "black")
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    ax.set_title(title, fontsize=9)
    if standalone:
        show_fig(title or "Row-normalised confusion matrix")


DATA = {"x": X_RAW_T, "y": Y_T, "idx": SPLITS["lesion"]}              # raw images, lesion-level split
DATA_ENH = {"x": X_ENH_T, "y": Y_T, "idx": SPLITS["lesion"]}          # enhanced images, same split

### 3.1b Sanity check: overfitting one tiny batch

Following the Week 6 tuning recipe, a fresh network is first trained on a handful of images with dropout and augmentation off, and the training loss must go close to zero. I use 14 images (2 per class) for the baseline CNN and for a randomly initialised ResNet18. The loss starts near ln 7 = 1.95 (1.93 and 2.08 below).

In [ ]:
def overfit_tiny_batch(arch, per_class=2, steps=80, lr=1e-3, seed=SEED):
    """Train a fresh network on a few images per class, dropout and augmentation off; returns the loss of every step."""
    set_seed(seed)
    tr = SPLITS["lesion"]["train"]
    pick = np.concatenate([tr[Y[tr] == c][:per_class] for c in range(N_CLASSES)])
    xb = normalise(X_RAW_T[torch.as_tensor(pick, device=device)].float() / 255)
    yb = Y_T[torch.as_tensor(pick, device=device)]
    model = build_model(arch, pretrained=False).to(device).train()
    for m in model.modules():                                                               # dropout would stop the loss reaching zero
        if isinstance(m, nn.Dropout):
            m.p = 0.0
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0)
    losses = []
    for _ in range(steps):
        opt.zero_grad(set_to_none=True)
        loss = F.cross_entropy(model(xb), yb)
        loss.backward(); opt.step()
        losses.append(loss.item())
    return losses


sanity = {arch: overfit_tiny_batch(arch) for arch in ("simple_cnn", "resnet18")}
fig, ax = plt.subplots(figsize=(6, 3.4))
for arch, l in sanity.items():
    ax.plot(l, label=f"{arch}: {l[0]:.2f} -> {l[-1]:.3f}")
ax.set_yscale("log"); ax.set_xlabel("step"); ax.set_ylabel("training loss"); ax.legend()
show_fig("Sanity check: training loss of fresh networks trained on 14 images (2 per class) with dropout and augmentation off.")
for arch, l in sanity.items():
    print(f"{arch:11s}: loss {l[0]:.3f} -> {l[-1]:.4f}")

### 3.2 Baseline architecture and settings

The baseline is a small CNN trained **from scratch**, kept compact (about 1.0 M parameters) because a deep network trained from scratch converges slowly and can run out of memory.

| Setting | Choice | Justification |
|---|---|---|
| Blocks | 5 × [Conv 3×3 → BatchNorm → ReLU → MaxPool 2×2], channels 32-64-128-256-256 | the pattern of Weeks 3–4 (small filters, more channels, smaller maps); a 224 px input becomes a 7×7 map after 5 poolings |
| Batch-norm | after every convolution | faster, more stable training that allows a larger learning rate (Ioffe & Szegedy, 2015; Week 4) |
| Output | global average pooling → dropout 0.3 → linear(7) | far fewer parameters than a flatten + dense head, so less over-fitting; dropout as regulariser (Srivastava et al., 2014; Week 4) |
| Optimiser | AdamW, lr 1e-3, weight decay 1e-4 | Adam is the lecture default (Kingma & Ba, 2015) and 1e-3 the Week 6 starting point for a new network; AdamW decouples weight decay (Loshchilov & Hutter, 2019) |
| Batch size | 64 | the lectures suggest 16–64 and small batches generalise well (Bengio, 2012; Masters & Luschi, 2018); 64 is the top of that range and where throughput on this GPU stops rising (Appendix B); §4.4 also tries 32 and 128 |
| Loss | cross-entropy, plain *or* class-weighted (both tested) | multi-class classification; the weights counter the 58 : 1 imbalance |
| LR schedule | ReduceLROnPlateau (×0.3, patience 2) on validation macro-F1 | Week 6 default: lowers the rate only when progress stalls |
| Stopping | early stopping on validation macro-F1 (patience 8), best epoch restored | prevents over-fitting; macro-F1 accounts for the imbalance |
| Precision | float16 mixed precision with gradient scaling | faster training at the same accuracy (Micikevicius et al., 2018; Appendix B) |
| Augmentation | §3.1 | label-preserving for dermatoscopy |

I train two variants, with **plain** and with **class-weighted** cross-entropy, and the one with the better *validation* macro-F1 becomes *the* baseline.

In [ ]:
# baseline CNN with plain and with class-weighted cross-entropy
cnn_stage = [{"name": "scratch", "epochs": EPOCHS_CNN, "lr": 1e-3, "train": "all", "head_mult": 1}]
W_TRAIN = inverse_frequency_weights(Y[SPLITS["lesion"]["train"]])
print("class weights (inverse frequency):", {c: round(float(w), 2) for c, w in zip(CLASSES, W_TRAIN.cpu())})

train_model("cnn_plain", "simple_cnn", DATA, pretrained=False, stages=cnn_stage, amp_dtype=AMP_FP16, patience=PATIENCE_CNN)
_ = train_model("cnn_weighted", "simple_cnn", DATA, pretrained=False, stages=cnn_stage, class_weights=W_TRAIN, amp_dtype=AMP_FP16,
            patience=PATIENCE_CNN)

In [ ]:
# the variant with the better validation macro-F1 becomes the baseline
plot_histories(["cnn_plain", "cnn_weighted"], "Baseline CNN: plain vs class-weighted loss (validation used for monitoring)")
val_table = pd.DataFrame([summary_row(n, evaluate(n, DATA, "val")[0]) for n in ["cnn_plain", "cnn_weighted"]]).set_index("model")
show_table(val_table.round(3), "The two baseline variants on the validation set.")
BASELINE = val_table["macro_f1"].idxmax()
print("baseline (better validation macro-F1):", BASELINE)

### 3.3 Baseline results

The next cell evaluates the chosen baseline on the **training, validation and test** sets. In Figure 12 the validation curves of both variants follow the training curves; for the weighted variant validation accuracy is even above training accuracy, because training accuracy is measured on augmented batches with dropout on.

In [ ]:
# the chosen baseline on train / validation / test, then per-class results, confusion matrix and a confidence interval
rows = []
for split in ["train", "val", "test"]:
    m = evaluate(BASELINE, DATA, split)[0]
    rows.append({"split": split, **{k: m[k] for k in ["accuracy", "balanced_accuracy", "macro_f1", "auc_macro", "mel_recall", "mel_specificity"]}})
base_table = pd.DataFrame(rows).set_index("split")
show_table(base_table.round(3), "The chosen baseline on the training, validation and test set.")

m_test, prob_test, y_test, _ = evaluate(BASELINE, DATA, "test")
rec = pd.DataFrame({"recall (sensitivity)": m_test["recall"], "specificity": m_test["specificity"], "precision": m_test["precision"], "f1": m_test["f1"],
                    "test images": np.bincount(y_test, minlength=N_CLASSES)}, index=CLASSES)
show_table(rec.round(3), "Baseline per-class sensitivity (recall), specificity, precision and F1 on the test set.")
plot_confusion(y_test, prob_test, f"Baseline ({BASELINE}), test set")
lo, hi = bootstrap_ci(y_test, prob_test.argmax(1), lambda a, b: f1_score(a, b, average="macro", zero_division=0))
print(f"baseline test macro-F1 = {m_test['macro_f1']:.3f}  (95 % bootstrap CI {lo:.3f} to {hi:.3f})")

# reference: a "classifier" that always answers nv, the majority class
ref_prob = np.full_like(prob_test, 1e-6); ref_prob[:, CLASS_IDX["nv"]] = 1.0
m_ref = compute_metrics(y_test, ref_prob)
ref_table = pd.DataFrame({"always answers nv": m_ref, "baseline CNN": m_test}).loc[["accuracy", "balanced_accuracy", "macro_f1", "mel_recall"]]
show_table(ref_table.astype(float).round(3), "Test-set reference: a classifier that always answers nv (majority class) against the baseline CNN.")

**Summary of the baseline.**

* **Under-fitting, not over-fitting.** Macro-F1 is 0.482 / 0.479 / 0.492 on train / validation / test and accuracy 0.673 / 0.654 / 0.673 (Table 8): a 1 M-parameter network trained from scratch for at most 40 epochs on about 7,000 images has not even learned its training images well. Validation and test agree, so choosing the variant on validation did not over-fit.
* **Accuracy hides the gain.** Test accuracy (0.673) is no better than always answering `nv` (0.669), but macro-F1 is 0.492 against 0.115 and melanoma recall 0.484 against 0 (Table 10).
* **Where it fails.** Recall is 45-48 % for `akiec`, `bcc`, `bkl`, `df` and `mel`, 76 % for `nv` and 80 % for `vasc` (20 test images). The up-weighted rare classes are predicted too often, so precision is low (14 % for `df`). 102 of 958 naevi are called melanoma, and only 77 of 159 melanomas are found.
* **What §4 tries.** Pretrained networks have learned edges, textures and colours from 1.2 million ImageNet images; the baseline has to learn them from about 7,000. Class weighting is a cheap first fix, and §4.2 compares it with balanced sampling on a stronger model.

## 4. Solution Improvement

I improve the baseline one step at a time. **Each step changes one thing** and is judged on the **validation** set; no decision uses the test set. Every experiment also gets a per-class table (recall, precision and F1), because the overall scores hide what happens in the rare classes.

| Step | Experiment | Section |
|---|---|---|
| A | **Transfer learning**: two-stage fine-tuning of pretrained models (against the from-scratch baseline in §4.5) | 4.1 |
| B | **Class imbalance**: nothing, weighted loss or balanced sampling | 4.2 |
| C | **Hair removal + CLAHE** against raw images | 4.3 |
| C2 | **Flips and large rotations** in the augmentation | 4.3b |
| D | **Learning rate, weight decay, batch size** (random search) | 4.4 |
| E | **Other pretrained models**: EfficientNet-B0 and ViT-B/16 against ResNet18 | 4.5 |
| F | **Run-to-run noise** (3 seeds) | 4.6 |
| G | **GAN-generated images** of the rare classes | 4.7 |
| H | Effect of the paper's **leaky split** | 4.8 |

### 4.1 Transfer learning: design and justification

I compare three architectures of different design and size, all with ImageNet-1k weights from torchvision:

| Model | Parameters | Why it is included |
|---|---|---|
| **ResNet18** (He et al., 2016) | 11.2 M | the image encoder of the reviewed paper; small and fast, so the natural reference; residual connections keep training stable (Week 4) |
| **EfficientNet-B0** (Tan & Le, 2019) | 4.0 M | a modern CNN found by architecture search and compound scaling, with different building blocks (depth-wise convolutions, squeeze-and-excitation) |
| **ViT-B/16** (Dosovitskiy et al., 2021) | 85.8 M | a transformer (Week 5), to test whether attention-based features transfer better to dermatoscopy; the paper also compares with ViT and Swin |

**Two-stage strategy (Weeks 4 and 6).** *Stage 1:* the pretrained weights are frozen and only the new 7-class output layer is trained (3 epochs, lr 1e-3); the BatchNorm layers of ResNet18 and EfficientNet-B0 still update their running statistics. *Stage 2:* the **whole** network is fine-tuned with a **much smaller learning rate**, 10× higher for the new layer. The Week 6 decision table recommends fine-tuning most of the network for a moderate-sized dataset that differs from ImageNet. Raghu et al. (2019) found that transfer to medical images helps mainly through a better starting point and faster convergence, and Yosinski et al. (2014) showed that transferred features still help after fine-tuning.

In [ ]:
def tl_stages(ft_lr, head_epochs=None, ft_epochs=None):
    """Stage 1 trains only the new classifier, stage 2 fine-tunes everything with a small learning rate."""
    return [{"name": "head", "epochs": head_epochs or EPOCHS_HEAD, "lr": 1e-3, "train": "head"},
            {"name": "finetune", "epochs": ft_epochs or EPOCHS_FINETUNE, "lr": ft_lr, "train": "all", "head_mult": 10}]


FT_LR_DEFAULT = 1e-4          # the usual fine-tuning range is 1e-5 to 1e-4 (Week 6)

### 4.2 Class imbalance: weighted loss and balanced sampling

I try three options on the same pretrained ResNet18, with everything else unchanged:

1. **Plain** cross-entropy.
2. **Weighted** cross-entropy with class weights $N / (K\,n_c)$, as in Lab 5.
3. **Balanced sampling**: every class is drawn equally often, so rare images are repeated, each time with a different augmentation.

In [ ]:
# three ways of handling the imbalance; the best on validation is used from here on (one fix, not both)
IMB_OPTIONS = {"plain": {}, "weighted": {"class_weights": W_TRAIN}, "sampler": {"balanced_sampler": True}}
for opt, kw in IMB_OPTIONS.items():
    train_model(f"r18_{opt}", "resnet18", DATA, pretrained=True, stages=tl_stages(FT_LR_DEFAULT), amp_dtype=AMP_FP16, **kw)

imb_names = [f"r18_{o}" for o in IMB_OPTIONS]
plot_histories(imb_names, "ResNet18: class-imbalance strategies (validation)")
imb_table = pd.DataFrame([summary_row(n, evaluate(n, DATA, "val")[0]) for n in imb_names]).set_index("model")
show_table(imb_table.round(3), "ResNet18 with three class-imbalance strategies on the validation set.")
IMB_BEST = imb_table["macro_f1"].idxmax().replace("r18_", "")
print("best imbalance strategy on validation:", IMB_BEST)

In [ ]:
def plot_class_recall(names, data, split="val", title=""):
    fig, ax = plt.subplots(figsize=(10, 3.6))
    w = 0.8 / len(names)
    for k, n in enumerate(names):
        ax.bar(np.arange(N_CLASSES) + k * w, evaluate(n, data, split)[0]["recall"], w, label=n)
    ax.set_xticks(np.arange(N_CLASSES) + 0.4 - w / 2, CLASSES); ax.set_ylabel("recall"); ax.set_ylim(0, 1.05)
    ax.set_title(title or f"Per-class recall on the {split} set"); ax.legend(fontsize=7); show_fig(title or f"Per-class recall on the {split} set")


plot_class_recall(imb_names, DATA, "val", "ResNet18: per-class recall on the validation set")
show_table(per_class_table(imb_names, DATA, labels=list(IMB_OPTIONS)).round(3),
           "ResNet18 with three class-imbalance strategies: per-class recall, precision and F1 on the validation set.")

**Result.** Without any remedy, validation accuracy is 0.819, macro-F1 0.670 and melanoma recall 0.509. The weighted loss gives 0.774 / 0.719 / 0.642 and the balanced sampler 0.784 / 0.724 / 0.642. Both remedies give up 3.5-4.5 points of accuracy for about +0.05 macro-F1 and +0.13 melanoma recall. They end within 0.005 of each other, which is about the seed noise measured in §4.6, so it is a tie; I keep the sampler because it scored slightly higher. Per class (Table 12), both remedies raise the recall of `akiec`, `df`, `vasc` and `mel` and lower that of `nv` (0.911 to 0.810-0.825). The macro-F1 gain comes from `akiec`, `df` and `vasc` (sampler F1 0.648, 0.778 and 0.800 against 0.433, 0.643 and 0.703); melanoma F1 does not improve (0.521 plain, 0.481 weighted, 0.498 sampler) because its precision falls from 0.533 to about 0.4.

### 4.3 Image enhancement (hair removal + CLAHE)

Same model, imbalance strategy and split; only the input images change (raw or enhanced). Since the run-to-run noise is around 0.005 (§4.6), a smaller change does not count as a gain, and the simpler raw pipeline is then kept.

In [ ]:
# the same recipe on the enhanced images, kept only if validation macro-F1 improves by at least 0.005
train_model(f"r18_{IMB_BEST}_enh", "resnet18", DATA_ENH, pretrained=True, stages=tl_stages(FT_LR_DEFAULT), amp_dtype=AMP_FP16,
            **IMB_OPTIONS[IMB_BEST])

enh_rows = [summary_row(f"raw images ({IMB_BEST})", evaluate(f"r18_{IMB_BEST}", DATA, "val")[0]),
            summary_row(f"hair removal + CLAHE ({IMB_BEST})", evaluate(f"r18_{IMB_BEST}_enh", DATA_ENH, "val")[0])]
enh_table = pd.DataFrame(enh_rows).set_index("model")
show_table(enh_table.round(3), "Raw against enhanced images (hair removal + CLAHE) on the validation set.")
show_table(per_class_table([f"r18_{IMB_BEST}", f"r18_{IMB_BEST}_enh"], [DATA, DATA_ENH], labels=["raw", "enhanced"]).round(3),
           "Raw against enhanced images: per-class recall, precision and F1 on the validation set.")
gain = enh_table["macro_f1"].iloc[1] - enh_table["macro_f1"].iloc[0]
USE_ENH = gain >= 0.005
DATA_BEST = DATA_ENH if USE_ENH else DATA
print(f"change in validation macro-F1: {gain:+.3f} -> using {'enhanced' if USE_ENH else 'raw'} images from here on")

**Result.** Hair removal + CLAHE lowers validation macro-F1 from 0.724 to 0.666 and melanoma recall from 0.642 to 0.553 (Table 13); balanced accuracy hardly changes (0.743 against 0.746) and AUC drops slightly (0.964 to 0.951). The loss is far beyond the 0.005 threshold, so the raw images are kept. Per class (Table 14), enhancement raises recall for `akiec`, `bcc` and `vasc` but lowers precision in six of the seven classes; the largest F1 losses are `df` (0.778 to 0.549) and `mel` (0.498 to 0.413).

### 4.3b Augmentation test: flips and large rotations

Dermatoscopic images have no fixed orientation (§3.1), but I still test this rather than assume it. Three settings are compared on the same ResNet18 recipe: **full** (the default: flips, rotation up to ±180°, zoom, shift, brightness and contrast), **mild** (no flips, rotation up to ±15°, weaker jitter) and **none**. Another setting replaces the default only if it beats it by at least 0.005 macro-F1.

In [ ]:
# three augmentation presets on the same recipe (the "full" run already exists from §4.2)
AUG_REF = f"r18_{IMB_BEST}_enh" if USE_ENH else f"r18_{IMB_BEST}"
for preset in ("mild", "none"):
    train_model(f"r18_aug_{preset}", "resnet18", DATA_BEST, pretrained=True, stages=tl_stages(FT_LR_DEFAULT),
                amp_dtype=AMP_FP16, aug=preset, **IMB_OPTIONS[IMB_BEST])

aug_names = [AUG_REF, "r18_aug_mild", "r18_aug_none"]
aug_table = pd.DataFrame([summary_row(n, evaluate(n, DATA_BEST, "val")[0]) for n in aug_names]).set_index("model")
aug_table.index = ["full (default)", "mild (no flips, rotation <= 15 deg)", "none"]
show_table(aug_table.round(3), "ResNet18 with three augmentation settings on the validation set.")
plot_histories(aug_names, "ResNet18 with three augmentation settings (validation)")
show_table(per_class_table(aug_names, DATA_BEST, labels=["full", "mild", "none"]).round(3),
           "ResNet18 with three augmentation settings: per-class recall, precision and F1 on the validation set.")

aug_f1 = dict(zip(("full", "mild", "none"), aug_table["macro_f1"]))
best_aug = max(aug_f1, key=aug_f1.get)
DEFAULT_AUG = best_aug if aug_f1[best_aug] - aug_f1["full"] >= 0.005 else "full"      # used by every later run
print("augmentation used from here on:", DEFAULT_AUG)

**Result.** Validation macro-F1 is 0.724 with the full augmentation, 0.714 with mild and 0.692 with none (Table 15), and melanoma recall falls from 0.642 to 0.478 and 0.421. Accuracy moves the other way (0.784, 0.818, 0.819) while balanced accuracy falls (0.743, 0.682, 0.658), so weaker augmentation favours the majority class. Neither weaker setting beats the default, so I keep the full augmentation. Per class (Table 16), dropping flips and large rotations raises `nv` recall (0.825 to 0.921 and 0.943) and lowers recall for `bcc`, `df` and `mel`. Melanoma F1 hardly changes (0.498, 0.490, 0.489) because its precision rises as its recall falls; the largest F1 drops are `bcc` (0.816 to 0.741 and 0.738) and, without augmentation, `df` (0.778 to 0.560).

### 4.4 Hyper-parameter search (random search)

Week 6 ranks the learning rate as the most important hyper-parameter, followed by weight decay and batch size. **Random search** tries more distinct values per hyper-parameter than a grid with the same budget (Bergstra & Bengio, 2012). Each trial is a short ResNet18 run (1 head and 3 fine-tuning epochs, same seed) with the learning rate log-uniform in 3e-5 … 1e-3, weight decay log-uniform in 1e-5 … 1e-2 and batch size 32, 64 or 128. Trial 0 is the default; the best **validation** macro-F1 wins.

In [ ]:
# random search; trial 0 is the default setting
from matplotlib.ticker import NullFormatter

rng = np.random.RandomState(SEED)
trials = [{"lr": float(10 ** rng.uniform(np.log10(3e-5), np.log10(1e-3))),
           "wd": float(10 ** rng.uniform(-5, -2)),
           "bs": int(rng.choice([32, 64, 128]))} for _ in range(N_SEARCH_TRIALS)]
trials[0] = {"lr": FT_LR_DEFAULT, "wd": 1e-4, "bs": BATCH_SIZE}

search_rows = []
for k, hp in enumerate(trials):
    name = f"r18_search{k}"
    train_model(name, "resnet18", DATA_BEST, pretrained=True, stages=tl_stages(hp["lr"], head_epochs=1, ft_epochs=SEARCH_FT_EPOCHS),
                amp_dtype=AMP_FP16, weight_decay=hp["wd"], batch_size=hp["bs"], patience=99, verbose=False, **IMB_OPTIONS[IMB_BEST])
    m = evaluate(name, DATA_BEST, "val")[0]
    search_rows.append({"trial": k, **hp, "val_macro_f1": m["macro_f1"], "val_acc": m["accuracy"], "val_bal_acc": m["balanced_accuracy"]})
search = pd.DataFrame(search_rows).set_index("trial")
show_table(search.round({"lr": 6, "wd": 6, "val_macro_f1": 3, "val_acc": 3, "val_bal_acc": 3}),
           "Random hyper-parameter search: learning rate, weight decay and batch size of every trial with its validation scores.")

BEST_HP = trials[int(search["val_macro_f1"].idxmax())]
print("best hyper-parameters:", {k: (round(v, 6) if isinstance(v, float) else v) for k, v in BEST_HP.items()})

fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
for ax, col, log in zip(axes, ["lr", "wd", "bs"], [True, True, False]):
    ax.scatter(search[col], search["val_macro_f1"], s=45, c=["#d95f02" if k == 0 else "#4c72b0" for k in search.index])
    for k in search.index:                                              # trial number next to each point
        ax.annotate(str(k), (search.loc[k, col], search.loc[k, "val_macro_f1"]), xytext=(5, 4), textcoords="offset points", fontsize=8)
    ax.set_xlabel({"lr": "learning rate", "wd": "weight decay", "bs": "batch size"}[col])
    ax.set_ylabel("val macro-F1")
    if log:
        ax.set_xscale("log")
        ax.xaxis.set_minor_formatter(NullFormatter())                  # otherwise the tick labels overlap
    else:
        ax.set_xticks([32, 64, 128]); ax.set_xlim(20, 140)             # show all three candidates
show_fig("Validation macro-F1 of every search trial against learning rate (left), weight decay (middle) and batch size (right). The numbers are trial numbers; orange is trial 0, the default.")

search_names = [f"r18_search{k}" for k in range(len(trials))]
show_table(per_class_table(search_names, DATA_BEST, labels=[f"trial {k}" for k in range(len(trials))], metrics=("f1",)).round(3),
           "Random search: per-class F1 of every trial on the validation set.")

**Result.** None of the random trials beats trial 0, the default (validation macro-F1 0.697; the other five range from 0.619 to 0.685), so the default is kept (Table 17). Batch size 128 was never drawn. The worst trial, trial 1, has the highest learning rate (4.6e-04). Per class (Table 18) no trial is best everywhere: trial 0 has the highest F1 only for `df` (0.788), and `df`, with 16 validation images, varies most between trials (0.490 to 0.788).

**Limit of the search.** All three hyper-parameters change at once in only six short trials, so a difference cannot be pinned on any single one; the search only shows that none of the tried settings beats the default. EfficientNet-B0 reuses this learning rate, while ViT-B/16 gets a smaller one.

### 4.5 Final pretrained models

All three models use the two-stage schedule and the chosen imbalance strategy and images. I fine-tune ViT-B/16 with lr 3e-5 (set by hand, not searched) in bfloat16, which has the range of float32 and needs no loss scaling; in the benchmark it was as fast as float16 for this model (Appendix B).